# **StanceEval2026**

### **Official Evaluation Script for the Arabic Stance Detection Shared Task**

This script is intended for organizers only.

It evaluates participant submissions against the hidden gold labels
for both the seen and unseen target tracks.

---

## **Expected Directory Structure**

```text
StanceEval26/Experiments/

├── data/
│   └── gold/
│       ├── test_seen_gold.csv
│       └── test_unseen_gold.csv
│
├── submissions/
│   ├── team1/
│   │   ├── submission_seen.csv
│   │   └── submission_unseen.csv
│   │
│   ├── team2/
│   │   ├── submission_seen.csv
│   │   └── submission_unseen.csv
│   │
│   └── ...
│
└── evaluation_outputs/

```



**Gold File Format**
id, target, stance

────────────────────────────────────────


**Submission File Format**
id, stance

────────────────────────────────────────

**Valid Stance Labels**

Against
Favor
None

────────────────────────────────────────

**Evaluation Metrics**

Official ranking metric:
Favg2 = macro-F1 over Favor and Against




In [ ]:

# ============================================================
# 1) Imports
# ============================================================

import os
import pandas as pd

from sklearn.metrics import f1_score, accuracy_score, classification_report

#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:

# ============================================================
# 2) Paths
# ============================================================

BASE_DIR = "./StanceEval26/Experiments"

GOLD_DIR = f"{BASE_DIR}/data/gold"
SUBMISSIONS_ROOT = f"{BASE_DIR}/submissions"
OUTPUT_ROOT = f"{BASE_DIR}/evaluation_outputs"

GOLD_SEEN_PATH = f"{GOLD_DIR}/test_seen_gold.csv"
GOLD_UNSEEN_PATH = f"{GOLD_DIR}/test_unseen_gold.csv"

os.makedirs(OUTPUT_ROOT, exist_ok=True)


# ============================================================
# 3) Settings
# ============================================================

ID_COL = "id"
TARGET_COL = "target"
LABEL_COL = "stance"

VALID_LABELS = ["Against", "Favor", "None"]

LABEL2ID = {
    "Against": 0,
    "Favor": 1,
    "None": 2,
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

#Targets
SEEN_TARGETS = ["Women Driving"]
UNSEEN_TARGETS = ["Ecars", "Trimester"]

TARGET_ORDER = ["Ecars", "Trimester", "Women Driving"]



In [ ]:

# ============================================================
# 4) Metrics
# ============================================================

def compute_metrics(y_true, y_pred):
    f_against = f1_score(
        y_true,
        y_pred,
        labels=[0],
        average="macro",
        zero_division=0
    )

    f_favor = f1_score(
        y_true,
        y_pred,
        labels=[1],
        average="macro",
        zero_division=0
    )

    f_none = f1_score(
        y_true,
        y_pred,
        labels=[2],
        average="macro",
        zero_division=0
    )

    favg2 = (f_favor + f_against) / 2.0
    favg3 = (f_favor + f_against + f_none) / 3.0
    acc = accuracy_score(y_true, y_pred)

    return {
        "F_favor": f_favor * 100,
        "F_against": f_against * 100,
        "F_none": f_none * 100,
        "Favg2": favg2 * 100,
        "Favg3": favg3 * 100,
        "Acc": acc * 100,
    }


def get_favg2(sub_df):
    if sub_df.empty:
        return None

    m = compute_metrics(
        sub_df["gold_id"],
        sub_df["pred_id"]
    )

    return round(m["Favg2"], 2)



In [ ]:

# ============================================================
# 5) Load and Validate Files
# ============================================================

def load_csv(path, file_name):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{file_name} not found: {path}")

    df = pd.read_csv(path, keep_default_na=False)
    df.columns = df.columns.astype(str).str.strip()

    return df


def validate_submission(gold_df, sub_df, split_name):
    gold_required_cols = [ID_COL, TARGET_COL, LABEL_COL]
    sub_required_cols = [ID_COL, LABEL_COL]

    for col in gold_required_cols:
        if col not in gold_df.columns:
            raise ValueError(f"Gold file for {split_name} is missing column: {col}")

    for col in sub_required_cols:
        if col not in sub_df.columns:
            raise ValueError(f"Submission file for {split_name} is missing column: {col}")

    for col in gold_required_cols:
        gold_df[col] = gold_df[col].astype(str).str.strip()

    for col in sub_required_cols:
        sub_df[col] = sub_df[col].astype(str).str.strip()

    invalid_gold_labels = sorted(set(gold_df[LABEL_COL]) - set(VALID_LABELS))
    if invalid_gold_labels:
        raise ValueError(
            f"Invalid labels found in gold file for {split_name}: {invalid_gold_labels}"
        )

    invalid_pred_labels = sorted(set(sub_df[LABEL_COL]) - set(VALID_LABELS))
    if invalid_pred_labels:
        raise ValueError(
            f"Invalid prediction labels found in submission for {split_name}: {invalid_pred_labels}"
        )

    if sub_df[ID_COL].duplicated().any():
        duplicated_ids = sub_df[sub_df[ID_COL].duplicated()][ID_COL].tolist()
        raise ValueError(
            f"Duplicate IDs found in {split_name} submission: {duplicated_ids[:10]}"
        )

    gold_ids = set(gold_df[ID_COL])
    sub_ids = set(sub_df[ID_COL])

    missing_ids = sorted(gold_ids - sub_ids)
    extra_ids = sorted(sub_ids - gold_ids)

    if missing_ids:
        raise ValueError(
            f"Missing predictions in {split_name} submission. "
            f"First missing IDs: {missing_ids[:10]}"
        )

    if extra_ids:
        raise ValueError(
            f"Extra IDs found in {split_name} submission. "
            f"First extra IDs: {extra_ids[:10]}"
        )



In [ ]:

# ============================================================
# 6) Evaluate One Split
# ============================================================

def evaluate_split(gold_path, submission_path, split_name, output_dir):
    gold_df = load_csv(gold_path, f"{split_name} gold file")
    sub_df = load_csv(submission_path, f"{split_name} submission file")

    validate_submission(gold_df, sub_df, split_name)

    merged_df = gold_df[[ID_COL, TARGET_COL, LABEL_COL]].merge(
        sub_df[[ID_COL, LABEL_COL]],
        on=ID_COL,
        how="inner",
        suffixes=("_gold", "_pred")
    )

    merged_df["gold_id"] = merged_df[f"{LABEL_COL}_gold"].map(LABEL2ID)
    merged_df["pred_id"] = merged_df[f"{LABEL_COL}_pred"].map(LABEL2ID)

    metrics = compute_metrics(
        merged_df["gold_id"],
        merged_df["pred_id"]
    )

    report = classification_report(
        merged_df["gold_id"],
        merged_df["pred_id"],
        labels=[0, 1, 2],
        target_names=["Against", "Favor", "None"],
        digits=4,
        zero_division=0
    )

    merged_path = f"{output_dir}/{split_name}_merged_predictions.csv"
    report_path = f"{output_dir}/{split_name}_classification_report.txt"

    merged_df.to_csv(merged_path, index=False)

    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report)

    return metrics, report, merged_df



In [ ]:


# ============================================================
# 7) Evaluate All Teams
# ============================================================

all_results = []
all_detailed_results = []

team_dirs = sorted([
    d for d in os.listdir(SUBMISSIONS_ROOT)
    if os.path.isdir(os.path.join(SUBMISSIONS_ROOT, d))
])

print("Found teams:")
for team in team_dirs:
    print("-", team)


for team_name in team_dirs:
    print("\n" + "=" * 70)
    print(f"Evaluating team: {team_name}")
    print("=" * 70)

    submission_dir = f"{SUBMISSIONS_ROOT}/{team_name}"

    submission_seen_path = f"{submission_dir}/submission_seen.csv"
    submission_unseen_path = f"{submission_dir}/submission_unseen.csv"

    team_output_dir = f"{OUTPUT_ROOT}/{team_name}"
    os.makedirs(team_output_dir, exist_ok=True)

    try:
        seen_metrics, seen_report, seen_merged = evaluate_split(
            GOLD_SEEN_PATH,
            submission_seen_path,
            split_name="seen",
            output_dir=team_output_dir
        )

        unseen_metrics, unseen_report, unseen_merged = evaluate_split(
            GOLD_UNSEEN_PATH,
            submission_unseen_path,
            split_name="unseen",
            output_dir=team_output_dir
        )

        test_merged = pd.concat(
            [unseen_merged, seen_merged],
            ignore_index=True
        )

        #  Leaderboard summary Favg2 table
        ec_df = test_merged[test_merged[TARGET_COL] == "Ecars"].copy()
        tr_df = test_merged[test_merged[TARGET_COL] == "Trimester"].copy()
        wd_df = test_merged[test_merged[TARGET_COL] == "Women Driving"].copy()

        unseen_overall_df = test_merged[
            test_merged[TARGET_COL].isin(UNSEEN_TARGETS)
        ].copy()

        seen_overall_df = test_merged[
            test_merged[TARGET_COL].isin(SEEN_TARGETS)
        ].copy()

        leaderboard_row = {
          "Team": team_name,

          # Per-target Favg2
          "EC_Favg2": get_favg2(ec_df),
          "TR_Favg2": get_favg2(tr_df),
          "WD_Favg2": get_favg2(wd_df),

          # Seen / Unseen / Test overall Favg2
          "Seen_Overall_Favg2": get_favg2(seen_overall_df),
          "Unseen_Overall_Favg2": get_favg2(unseen_overall_df),
          "Test_Overall_Favg2": get_favg2(test_merged),
      }

        all_results.append(leaderboard_row)

        # Detailed metrics table
        detailed_row = {
            "Team": team_name,

            "EC_Favg2": get_favg2(ec_df),
            "TR_Favg2": get_favg2(tr_df),
            "Unseen_Overall_Favg2": get_favg2(unseen_overall_df),
            "WD_Favg2": get_favg2(wd_df),
            "Seen_Overall_Favg2": get_favg2(seen_overall_df),
            "Test_Overall_Favg2": get_favg2(test_merged),

            "Seen_Favg2": round(seen_metrics["Favg2"], 2),
            "Seen_Favg3": round(seen_metrics["Favg3"], 2),
            "Seen_Acc": round(seen_metrics["Acc"], 2),

            "Unseen_Favg2": round(unseen_metrics["Favg2"], 2),
            "Unseen_Favg3": round(unseen_metrics["Favg3"], 2),
            "Unseen_Acc": round(unseen_metrics["Acc"], 2),
        }

        all_detailed_results.append(detailed_row)

        print("Evaluation completed successfully.")

        print("\nSeen Classification Report")
        print(seen_report)

        print("\nUnseen Classification Report")
        print(unseen_report)

    except Exception as e:
        print(f"Error evaluating {team_name}:")
        print(e)



In [ ]:
# ============================================================
# 8) Leaderboard Ranking
# ============================================================
# Results are ranked using Favg2, computed as the
# macro-average F1-score over Favor and Against.

# Per-target Favg2 scores are reported separately.
# Seen_Overall_Favg2, Unseen_Overall_Favg2, and
# Test_Overall_Favg2 are computed over all instances in their corresponding subsets.

# ============================================================

leaderboard_df = pd.DataFrame(all_results)

if not leaderboard_df.empty:
    leaderboard_df = leaderboard_df.sort_values(
        by="Test_Overall_Favg2",
        ascending=False
    ).reset_index(drop=True)

    leaderboard_path = f"{OUTPUT_ROOT}/leaderboard_results.csv"
    leaderboard_df.to_csv(leaderboard_path, index=False)

    print("\nLeaderboard Results")
    display(leaderboard_df)

    print("\nSaved results to:")
    print(leaderboard_path)
else:
    print("\nNo valid leaderboard results found.")